<a href="https://colab.research.google.com/github/Gaurav9571/week1_Gaurav_Mittal/blob/main/WEEK7_GAURAV_MITTAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# Install required libraries
!pip -q install -U \
langchain \
langchain-community \
langchain-google-genai \
langchain-text-splitters \
pypdf \
faiss-cpu \
sentence-transformers

In [21]:
import os

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI

In [44]:
import os
import getpass

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API Key: ")

Enter API Key: ··········


In [23]:
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded:", pdf_path)

Saving M.L. UNIT 01.pdf to M.L. UNIT 01 (2).pdf
Uploaded: M.L. UNIT 01 (2).pdf


In [45]:
loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 22


In [46]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 73


In [47]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [48]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


In [49]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

In [50]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

In [51]:
question = input("Ask a Question: ")

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

Ask a Question: give summary and some questions


In [52]:
prompt = f"""
You are a helpful AI assistant.

Answer ONLY from the provided context.

If the answer is not available,
say:

"I couldn't find this information in the document."

Context:

{context}

Question:

{question}

Answer:
"""

response = llm.invoke(prompt)

print(response.content)

I couldn't find this information in the document.
